In [3]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/marketing_campaign.csv', sep="\t") # https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis/data
Year = df['Year_Birth']
df.dtypes # 3 строки формата str, проверить вручную

ID                       int64
Year_Birth               int64
Education                  str
Marital_Status             str
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer                str
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object

In [6]:
pd.set_option('display.max_columns', 30)
# pd.set_option('display.float_format', '{:3f}'.format)
df.describe()

# Имеем:
# Нереальный минимальный год рождения;
# Пропуски в Income;
# Нереальный max в Income;
# Неизменные Z_CostContact и Z_Revenue (мусор или константы)

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714,26.302232,166.950000,37.525446,27.062946,44.021875,2.325000,4.084821,2.662054,5.790179,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393,39.773434,225.715373,54.628979,41.280498,52.167439,1.932238,2.778714,2.923101,3.250958,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,1.000000,9.000000,1.000000,2.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,8.000000,24.000000,2.000000,4.000000,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,33.000000,56.000000,3.000000,6.000000,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,263.000000,362.000000,15.000000,27.000000,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000


In [7]:
df = df.drop(columns=['Z_CostContact', 'Z_Revenue']) # Удаляем столбцы, которые не пригодятся в анализе

In [8]:
# df['Year_Birth'].sort_values() # видим 3 юзера с нереальным годом рождения (<= 1900), ограничиваем по году рождения (> 1900)

df = df[df['Year_Birth'] > 1900]

In [9]:
# df['Income'].sort_values(ascending=False) # Имеем одно нереальное значение, ограничиваем по нему

df = df[df['Income'] < 666666]

# [... < 666666] уже удалил все NaN, для наглядности можно написать: df = df[(df['Income'] < 666666) & (df['Income'].notna())]

In [10]:
# df['Education'].value_counts() # в порядке
# df['Marital_Status'].value_counts() # имеем 2 бессмысленные категории, одну ошибочную.

df = df[~df['Marital_Status'].isin(['Absurd', 'YOLO'])] # убираем лишние категории
df.loc[df['Marital_Status'] == 'Alone', 'Marital_Status'] = 'Single' # исправляем ошибочную категорию

In [11]:
# df['Dt_Customer'].value_counts() # Дата в формате строки, изменить.

df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
# df['Dt_Customer'].sort_values() # в порядке

In [12]:
df.describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
count,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000,2208.000000
mean,5584.532609,1968.904438,51943.520833,0.442482,0.505888,2013-07-10 12:50:13.043478,49.057518,305.226902,26.298007,167.004076,37.527174,27.065217,43.782609,2.322464,4.086051,2.669384,5.805707,5.322464,0.073822,0.074275,0.072464,0.063859,0.013587,0.009058,0.149909
min,0.000000,1940.000000,1730.000000,0.000000,0.000000,2012-07-30 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2814.750000,1959.000000,35196.000000,0.000000,0.000000,2013-01-16 00:00:00,24.000000,24.000000,1.750000,16.000000,3.000000,1.000000,9.000000,1.000000,2.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5454.500000,1970.000000,51371.000000,0.000000,0.000000,2013-07-08 12:00:00,49.000000,174.000000,8.000000,68.000000,12.000000,8.000000,24.000000,2.000000,4.000000,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8418.500000,1977.000000,68487.000000,1.000000,1.000000,2013-12-31 00:00:00,74.000000,505.500000,33.000000,232.250000,50.000000,33.000000,56.000000,3.000000,6.000000,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,11191.000000,1996.000000,162397.000000,2.000000,2.000000,2014-06-29 00:00:00,99.000000,1493.000000,199.000000,1725.000000,259.000000,262.000000,321.000000,15.000000,27.000000,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,3246.084196,11.697572,21536.756816,0.537112,0.544330,NaN,28.935531,337.606819,39.731780,224.316242,54.578404,41.111454,51.513978,1.924304,2.743171,2.925185,3.253777,2.423679,0.261541,0.262278,0.259313,0.244556,0.115795,0.094763,0.357063


In [13]:
conn = sqlite3.connect('../data/marketing.db')
df.to_sql('customers', conn, if_exists='replace', index=False)

q25 = df['Income'].quantile(0.25)
q50 = df['Income'].quantile(0.5)
q75 = df['Income'].quantile(0.75)

In [20]:
# Поиск зависимостей от семейного статуса
query = """
SELECT
    Marital_Status,
    AVG(Response) as response_rate
FROM
    customers
GROUP BY
    Marital_Status
ORDER BY
    response_rate DESC
"""

pd.read_sql_query(query, conn).round(4)

,Marital_Status,response_rate
0,Widow,0.2368
1,Single,0.2262
2,Divorced,0.2078
3,Married,0.1144
4,Together,0.1051


In [21]:
# Поиск зависимостей от возрастной группы
query = '''
SELECT
    CASE
        WHEN 2026 - Year_Birth < 30 THEN 'Младше 30 лет'
        WHEN 2026 - Year_Birth BETWEEN 30 AND 45 THEN '30 - 45 лет'
        WHEN 2026 - Year_Birth BETWEEN 46 AND 60 THEN '46 - 60 лет'
        ELSE 'Старше 60 лет'
    END as age_group,
    AVG(Response) as response_rate
FROM
    customers
GROUP BY
    age_group
ORDER BY
    response_rate DESC

''' 

pd.read_sql_query(query, conn).round(4)

,age_group,response_rate
0,30 - 45 лет,0.1979
1,46 - 60 лет,0.1402
2,Старше 60 лет,0.1396


In [22]:
# Поиск зависимостей от достатка
query = f'''
SELECT 
    CASE
        WHEN Income < {q25} THEN 'низкий (ниже {q25})'
        WHEN Income BETWEEN {q25} AND {q50} THEN 'средний (между {q25} и {q50})'
        WHEN Income BETWEEN {q50} AND {q75} THEN 'выше среднего (между {q50} и {q75})'
        ELSE 'высокий (больше {q75})'
    END as income_group,
    AVG(Response) as response_rate
FROM
    customers
GROUP BY
    income_group
ORDER BY
    response_rate DESC
''' 

pd.read_sql_query(query, conn).round(4)

,income_group,response_rate
0,высокий (больше 68487.0),0.2686
1,средний (между 35196.0 и 51371.0),0.1248
2,низкий (ниже 35196.0),0.1034
3,выше среднего (между 51371.0 и 68487.0),0.1031


In [23]:
# Процент успеха каждой компании
query = '''
SELECT
    AVG(AcceptedCmp1) as 'Campaign 1 Success rate',
    AVG(AcceptedCmp2) as 'Campaign 2 Success rate',
    AVG(AcceptedCmp3) as 'Campaign 3 Success rate',
    AVG(AcceptedCmp4) as 'Campaign 4 Success rate',
    AVG(AcceptedCmp5) as 'Campaign 5 Success rate'
FROM
    customers

'''

pd.read_sql_query(query, conn).T.round(4).sort_values(by=0, ascending=False)

,0
Campaign 4 Success rate,0.0743
Campaign 3 Success rate,0.0738
Campaign 5 Success rate,0.0725
Campaign 1 Success rate,0.0639
Campaign 2 Success rate,0.0136


In [ ]:
# Траты в зависимости от кол-ва детей
query = '''
SELECT
    Kidhome + Teenhome as amount_of_kids,
    AVG(MntWines + MntFruits + MntMeatProducts + MntFishProducts + MntSweetProducts + MntGoldProds) as average_spendings
FROM
    customers
GROUP BY
    amount_of_kids
'''

pd.read_sql_query(query, conn).round(4)

,amount_of_kids,average_spendings
0,0,1103.7841
1,1,475.9379
2,2,246.7356
3,3,255.5000


In [25]:
# Зависимость трат клиентов от времени регистрации
query = '''
SELECT
    CASE
        WHEN (julianday((SELECT MAX(Dt_Customer) FROM customers)) - julianday(Dt_Customer)) < 30 THEN '<30'
        WHEN (julianday((SELECT MAX(Dt_Customer) FROM customers)) - julianday(Dt_Customer)) BETWEEN 31 AND 180 THEN '30-180'
        WHEN (julianday((SELECT MAX(Dt_Customer) FROM customers)) - julianday(Dt_Customer)) BETWEEN 181 AND 360 THEN '181-360'
        ELSE '360+'
    END as days_registered_group,
    COUNT(*) as number_of_clients,
    AVG(MntWines + MntFruits + MntMeatProducts + MntFishProducts + MntSweetProducts + MntGoldProds) as average_spendings
FROM
    customers
GROUP BY
    days_registered_group
ORDER BY
        CASE days_registered_group
        WHEN '<30' THEN 1
        WHEN '30-180' THEN 2
        WHEN '181-360' THEN 3
        WHEN '360+' THEN 4
    END
'''

pd.read_sql_query(query, conn)

,days_registered_group,number_of_clients,average_spendings
0,<30,76,468.381579
1,30-180,477,500.280922
2,181-360,572,545.977273
3,360+,1083,695.765466
